In [ ]:
from statsbombpy import sb
import pandas as pd

def get_season_matches_ids(df):
    ids = []
    for index, row in df.iterrows():
        if row['match_status'] == 'available':
            ids.append(row['match_id'])
    return ids

def get_matches(c_id, s_id):
    matches = sb.matches(competition_id=c_id, season_id=s_id)
    return matches

def get_events(event_types, m_id):
    df = []
    events = sb.events(match_id=m_id, split=True, flatten_attrs=False)
    for event_type in event_types:
        if len(events[event_type]):
            df.append(events[event_type])
    return df

def get_event_json_col_name(event_type):
    return event_type.lower().replace(' ', '_').replace('*', '')

def get_types_without_json_column(events_dfs):
    types_without_json_column = set()
    for event_df in events_dfs:
        event_type = event_df['type'][0]
        try:
            # check if event_df has json event column
            event_df[get_event_json_col_name(event_type)][0]
        except:
            # if not add type to set
            types_without_json_column.add(event_type)
    return types_without_json_column

def get_merged_events(events_types, match_id):
    events_dfs = get_events(events_types, match_id)
    types_without_json_column = get_types_without_json_column(events_dfs)
    events_dfs_renamed = []
    for event_df in events_dfs:
        event_type = event_df['type'][0]
        if event_type not in types_without_json_column:
            events_dfs_renamed.append(event_df.rename(columns={get_event_json_col_name(event_type) : 'event'}))
        else:
            events_dfs_renamed.append(event_df)
    events = pd.concat(events_dfs_renamed, ignore_index=True)
    events = events.fillna('')
    return events.sort_values('index', ignore_index=True)

def get_sb360(match_id):
    return sb.frames(match_id, fmt='dataframe')

def add_context(events, frames):
    newEvents = events
    context_locations = [frames.loc[frames['id'] == event[0]]['location'].values for event in events.values]
    teammate_locations = [frames.loc[frames['id'] == event[0]]['teammate'].values for event in events.values]
    actor_locations = [frames.loc[frames['id'] == event[0]]['actor'].values for event in events.values]
    newEvents['context_locations'] = context_locations
    newEvents['teammate_locations'] = teammate_locations
    newEvents['actor_locations'] = actor_locations
    return newEvents

def get_events_from_matches(events_types, match_ids):
    matches = []
    for id in match_ids:
        frames = get_sb360(id)
        events = get_merged_events(events_types, id)
        events = add_context(events, frames)
        matches.append(events)
    return pd.concat(matches, ignore_index=True)

In [ ]:
EVENTS_TO_ANALIZE = ['passes', 'carrys', 'ball_receipts', 'ball_recoverys', 'dribbles', 'blocks', 'clearances', 'duels',
'dispossesseds', 'errors', 'foul_committeds', 'foul_wons', 'half_ends', 'injury_stoppages', 'interceptions', 'miscontrols', 'shots']
match_ids = get_season_matches_ids(get_matches(38, 235))
euro_matches = get_events_from_matches(EVENTS_TO_ANALIZE, match_ids)
euro_matches.to_csv('ekstraklasa_matches.csv')